# 04 · GraphRAG head-to-head

Graph retrieval offers another way to find a connecting passage: follow a
relationship represented in an index rather than ask an agent to formulate the
next search. Comparing this approach fairly requires the same original sources,
question set, final answerer, and reader budget, as well as a clear record of
which graph implementation and index produced the ranking.

The offline comparison traverses a small hand-authored passage graph and uses
the real source-based answering adapter. Removing a bridge or changing its seed
shows how connectivity affects the evidence reaching the reader. This is a toy
graph, not HippoRAG 2 or LightRAG. The artifact check and cost worksheet explain
how corpus identity, passage mapping, and build cost affect a graph comparison.

## 1. Graph retrieval and source evidence

IRCoT adds an answering/retrieval loop over a dense index. A graph contrast
instead changes how connected source passages are found. Keep the corpus,
question IDs, final answer model/template, and five-passage reader budget fixed.

`question → retrieval adapter → ranked original passage IDs → same five-passage answerer`

| Layer | What to inspect |
| --- | --- |
| Index construction | Extraction model, source revision, passage map, checksum, build tokens |
| Query retrieval | Backend/variant, traversal settings, candidate ranking, query calls |
| Final generation | Original source text, fixed evidence budget, source citations |

HippoRAG 2's paper configuration and the shared atlas-rag variant need distinct
labels. LightRAG's returned chunks must map back to original MuSiQue passages
and be deduplicated before spending the passage budget. Entity summaries alone
are not a replacement for original source evidence.

## 2. A worked example

### Make the passage connection visible

The graph below is hand-authored over the same invented corpus as notebook 03:
`demo::00 (Mira wrote Beacon) → demo::10 (Mira was born in Oslo)`.

**Predict first:** dense retrieval initially favors the book passage. What changes
when the graph admits the birthplace passage near the top of the ranking?

This toy traversal is real Python graph traversal, but it is **not** HippoRAG 2
or LightRAG. The seed and edge are supplied by us. It isolates passage
connectivity and reader coverage, not entity-linking quality, extraction quality,
or graph construction cost.

In [1]:
# ruff: noqa: E402 - resolve checkout imports before the first example
import sys
from pathlib import Path


# True runs each model step on MODELS through the proxy and caches the result.
# False replays the reference results from RI #2's release with no model calls.
RUN_EXPENSIVE = False

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "aieng-advanced-rag/aieng").is_dir())
sys.path[:0] = [str(ROOT / "aieng-advanced-rag"), str(ROOT / "implementations/cross_document_synthesis")]
from synthesis_lab.tutorial import read_result, require_compute, show_table


require_compute()

from aieng.advanced_rag.models import choose_models


MODELS = choose_models(show=True)

from tempfile import TemporaryDirectory

from aieng.advanced_rag.cache import CacheKey, TieredResultCache
from synthesis_lab.evaluation import question_scores
from synthesis_lab.graph import GraphAnswerer, PendingGraphArtifact, resolve_index
from synthesis_lab.methods import DenseAnswerer
from synthesis_lab.tutorial import source_answer, synthetic_corpus


corpus, vectors, encode = synthetic_corpus()
question = corpus.questions[0]
show_table(
    ["ID", "Invented source text"],
    [[p.passage_id, p.text] for p in corpus.passages if p.passage_id in question.supporting_ids],
)


async def toy_graph_answer(*, bridge=True, edge_steps=1, seed="demo::00"):
    """Walk an invented passage graph, then use the real shared final answerer."""
    edges = {"demo::00": ["demo::10"]} if bridge else {}

    async def rank(query, k):
        visited, frontier = [seed], [seed]
        for _ in range(edge_steps):
            next_ids = [target for source in frontier for target in edges.get(source, []) if target not in visited]
            visited.extend(dict.fromkeys(next_ids))
            frontier = next_ids
        # Fill unused candidate slots with the same distractor order.
        return (visited + [p.passage_id for p in corpus.passages if p.passage_id not in visited])[:k]

    adapter = GraphAnswerer(corpus.passages, rank, source_answer)
    answer = await adapter.aanswer(question.question_id, question.question)
    return answer, adapter.traces[question.question_id]


def scored_row(label, answer, generation_ids):
    """Score after answering; gold annotations never enter either adapter."""
    metrics = question_scores(
        question,
        {
            "answer": answer.answer.text,
            "abstained": answer.abstained,
            "retrieved_ids": list(answer.retrieved_ids),
            "generation_ids": generation_ids,
            "citation_ids": list(answer.answer.chunk_ids),
        },
    )
    return [
        label,
        answer.answer.text or "(abstain)",
        metrics["token_f1"],
        metrics["recall_at_5"],
        metrics["generation_all_supports"],
        generation_ids,
    ]


dense = DenseAnswerer(corpus.passages, vectors, encode, source_answer)
dense_answer = dense.answer(question.question_id, question.question)
graph_answer, graph_trace = await toy_graph_answer()
missing_answer, missing_trace = await toy_graph_answer(bridge=False)
show_table(
    ["Synthetic method", "Answer", "F1", "Recall@5", "Generation all-supports", "Generation IDs"],
    [
        scored_row("Dense fixture", dense_answer, dense.traces[question.question_id]["generation_ids"]),
        scored_row("Toy graph: bridge", graph_answer, graph_trace["generation_ids"]),
        scored_row("Toy graph: missing bridge", missing_answer, missing_trace["generation_ids"]),
    ],
)
assert graph_answer.answer.text == "Oslo" and missing_answer.abstained

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget unknown (the proxy did not answer)


| ID | Invented source text |
| --- | --- |
| demo::00 | Mira wrote Beacon. |
| demo::10 | Mira was born in Oslo. |

| Synthetic method | Answer | F1 | Recall@5 | Generation all-supports | Generation IDs |
| --- | --- | --- | --- | --- | --- |
| Dense fixture | Not stated in the provided sources. | 0.0000 | 0.5000 | 0.0000 | ['demo::00', 'demo::01', 'demo::02', 'demo::03', 'demo::04'] |
| Toy graph: bridge | Oslo | 1.0000 | 1.0000 | 1.0000 | ['demo::00', 'demo::10', 'demo::01', 'demo::02', 'demo::03'] |
| Toy graph: missing bridge | Not stated in the provided sources. | 0.0000 | 0.5000 | 0.0000 | ['demo::00', 'demo::01', 'demo::02', 'demo::03', 'demo::04'] |

**Expected observation:** both methods use the same final source-checking
completion. The graph with the bridge exposes both facts early enough to fit
the reader budget; the missing bridge produces an abstention. One edge traversal
here connects two supporting facts. Graph traversal depth and annotated question
hop count are different quantities.

The example guarantees a useful edge and seed. A production graph can omit,
misconnect, or hallucinate relations; its gain cannot be inferred from this toy.

### Exercise the artifact boundary without downloading an index

An empty temporary artifact cache demonstrates the resolver's input validation.
The resolver checks archive/tree checksums, corpus revision/digest,
complete ordered passage mapping, producer revision, and backend settings.

In [2]:
with TemporaryDirectory() as directory:
    temporary = Path(directory)
    empty_cache = TieredResultCache(local_dir=temporary / "local", global_dir=temporary / "global")
    key = CacheKey("ri4", "index", "learning-fixture", {})
    try:
        resolve_index(empty_cache, key, corpus, temporary / "unpacked")
    except PendingGraphArtifact as error:
        print("Expected pending dependency:", error)

Expected pending dependency: RI #4 prebuilt index is absent from the artifact cache


## 3. Recorded results

Dense, atlas-rag HippoRAG 2, and LightRAG have results on all 1,000 MuSiQue
questions over complete indexes. HippoRAG 2 is atlas-rag's entity/relation
configuration, not the authors' paper implementation. LightRAG runs in `mix`
mode over RI #4's release `reference-ri4-bd05f7bb16c6`, which indexes all
11,656 passages. The answer template and five-passage reader are shared.

### Evaluation data and fixed configuration

The table below identifies the saved **dense generation reference**, not a measured graph backend.
The MuSiQue reference searches all 11,656 passages for 1,000 original questions.
The NQ control searches 9,633 passages for its 1,000 questions. Model, encoder,
source revision, prompt identity, and evidence settings travel with the saved
reports. Loading these records makes no model calls and does not recompute the
measurements. New experiments must keep the controls fixed or identify the change.


In [3]:
saved = read_result("evaluation/musique_dense.json")
spec = saved["spec"]
assert saved["completed"] and saved["coverage_complete"] and len(saved["records"]) == 1000
show_table(
    ["Recorded setting", "Value"],
    [
        ["Answer model", spec["model"]],
        ["Generation endpoint", spec["base_url"]],
        ["Local encoder", spec["embedding"][0]],
        ["Encoder revision", spec["embedding"][1]],
        ["Corpus revision", spec["provenance"]["revision"]],
        ["Final passage budget", spec["bounds"]["generation_passages"]],
    ],
)
print("Answer-template SHA-256:", spec["answer_prompt_sha256"])
print("Saved configuration SHA-256:", saved["spec_sha256"])

| Recorded setting | Value |
| --- | --- |
| Answer model | Qwen3_6-35B-A3B |
| Generation endpoint | https://proxy.vectorinstitute.ai/v1 |
| Local encoder | BAAI/bge-small-en-v1.5 |
| Encoder revision | 5c38ec7c405ec4b44b94cc5a9bb96e735b38267a |
| Corpus revision | 5ec05b38deecc3318bb432c69865959c56058990 |
| Final passage budget | 5 |

Answer-template SHA-256: 517c503f1380aa40f6d20b1fc0b8f6538bbf367601e79d4bca0d316cea6eea0f
Saved configuration SHA-256: 40044ef66097a00af9d062567b74309de692c301c73ea0355ccdb6d3ff7ca620


In [4]:
summary = read_result("evaluation/graphrag/summary.json")
labels = {"dense": "Dense", "hipporag2": "HippoRAG 2 (atlas-rag)", "lightrag": "LightRAG (mix)"}
show_table(
    ["Full-index method", "N", "EM", "F1", "Recall@5", "Generation all-supports", "Failures"],
    [
        [
            labels[r["method"]],
            r["scores"]["overall"]["questions"],
            *[
                r["scores"]["overall"]["metrics"][key]
                for key in ("exact_match", "token_f1", "recall_at_5", "generation_all_supports", "failure")
            ],
        ]
        for r in summary["reports"]
        if r["corpus"] == "musique" and r["method"] in labels
    ],
)
show_table(
    ["Graph method minus dense", "Paired F1 difference", "95% interval", "Added live calls"],
    [
        [
            labels[r["method"]],
            r["differences"]["token_f1"]["mean"],
            f"[{r['differences']['token_f1']['low']:.4f}, {r['differences']['token_f1']['high']:.4f}]",
            r["added_calls"],
        ]
        for r in summary["comparisons"]
        if r["method"] in {"hipporag2", "lightrag"} and "wording" not in r
    ],
)
for name in ("hipporag2", "lightrag"):
    print(name, "index coverage:", summary["artifact_validation"][name]["index_coverage"])

| Full-index method | N | EM | F1 | Recall@5 | Generation all-supports | Failures |
| --- | --- | --- | --- | --- | --- | --- |
| Dense | 1000 | 0.2010 | 0.2733 | 0.5060 | 0.1710 | 0.0000 |
| HippoRAG 2 (atlas-rag) | 1000 | 0.3140 | 0.4073 | 0.6254 | 0.3230 | 0.0000 |
| LightRAG (mix) | 1000 | 0.1950 | 0.2655 | 0.4663 | 0.1370 | 0.0000 |

| Graph method minus dense | Paired F1 difference | 95% interval | Added live calls |
| --- | --- | --- | --- |
| HippoRAG 2 (atlas-rag) | 0.1340 | [0.1088, 0.1600] | 1998 |
| LightRAG (mix) | -0.0077 | [-0.0236, 0.0079] | 1998 |

hipporag2 index coverage: {'complete': True, 'represented_passages': 11656}
lightrag index coverage: {'complete': True, 'represented_passages': 11656, 'missing_passage_ids': [], 'allow_incomplete': False}


RI #4 records its own graph test and construction costs. Those are producer evidence;
they do not fill an RI #2 head-to-head cell until corpus mapping, backend variant,
generation settings, and matched question coverage pass this experiment's checks.
See [the RI #4 materials](../graphrag_technical_support/README.md).

For a real result, report build tokens/time separately from query tokens/time.
Unknown construction cost stays unknown. Amortized cost depends on how many
queries reuse the index; warm cache hits do not establish a zero-cost build.

### Construction cost from RI #4's producer records

These compact records describe the completed producer builds over 11,656
passages. They are separate from RI #2 query measurements and do not validate a
future archive's identity. HippoRAG 2 is explicitly the atlas-rag variant.
LightRAG's build was resumed, so a single elapsed build time is not recorded.
Each row carries a pinned source link; interpreting cache reuse and failed work
still requires the producer's cost notes. Build cost alone does not establish query quality.


In [5]:
producer = read_result("examples/graph_build_costs.json")
print(producer["note"])
show_table(
    [
        "Producer implementation",
        "Passages",
        "Extraction model",
        "Recorded calls",
        "Recorded tokens",
        "Build wall (hours)",
    ],
    [
        [
            r["implementation"],
            r["passages"],
            r["model"],
            r["calls"],
            r["total_tokens"],
            r["wall_seconds"] / 3600 if r["wall_seconds"] is not None else "not recorded",
        ]
        for r in producer["records"]
    ],
)
for build in producer["records"]:
    print(build["implementation"], "source:", build["source_url"])

Recorded construction usage belongs to RI4 producer builds. LightRAG has no single elapsed build time across resumed sessions. Artifact identity and RI2 query performance remain unverified. These totals are not complete billing estimates.


| Producer implementation | Passages | Extraction model | Recorded calls | Recorded tokens | Build wall (hours) |
| --- | --- | --- | --- | --- | --- |
| atlas-rag variant | 11656 | Qwen3_6-35B-A3B | 11285 | 7422640 | 3.9290 |
| LightRAG | 11656 | Qwen3_6-35B-A3B | 28198 | 78734144 | not recorded |

atlas-rag variant source: https://github.com/VectorInstitute/advanced-RAG/blob/e2994b28de0914ee4b066edc482a2b2ab3adf0e0/implementations/graphrag_technical_support/outputs/musique_hipporag2_index_cost.json
LightRAG source: https://github.com/VectorInstitute/advanced-RAG/blob/e2994b28de0914ee4b066edc482a2b2ab3adf0e0/implementations/graphrag_technical_support/outputs/musique_lightrag_index_cost.json


## 4. Experiments

Edit these controls independently. **Hypotheses:** removing the bridge or setting
`EDGE_STEPS=0` loses the second fact; a wrong seed can fail despite a correct edge.
Change one control, predict the generation IDs, then inspect the result.
The final source completion and reader budget remain fixed.

In [6]:
INCLUDE_BRIDGE = True
EDGE_STEPS = 1
SEED_ID = "demo::00"  # Try "demo::15" to model a bad seed.
variant, variant_trace = await toy_graph_answer(bridge=INCLUDE_BRIDGE, edge_steps=EDGE_STEPS, seed=SEED_ID)
show_table(
    ["Synthetic method", "Answer", "F1", "Recall@5", "Generation all-supports", "Generation IDs"],
    [scored_row("Edited toy graph", variant, variant_trace["generation_ids"])],
)

| Synthetic method | Answer | F1 | Recall@5 | Generation all-supports | Generation IDs |
| --- | --- | --- | --- | --- | --- |
| Edited toy graph | Oslo | 1.0000 | 1.0000 | 1.0000 | ['demo::00', 'demo::10', 'demo::01', 'demo::02', 'demo::03'] |

### Work through a build/query cost estimate

These numbers are editable **hypothetical costs**, not graph benchmark output.
Use the worksheet to ask when amortized index cost becomes smaller than an
iterative query policy. A lower cost does not establish acceptable quality.

In [7]:
BUILD_TOKENS = 1_000_000
GRAPH_QUERY_TOKENS = 1_000
ITERATIVE_QUERY_TOKENS = 4_000
QUERY_COUNTS = [10, 1_000, 10_000]
show_table(
    ["Queries", "Graph total tokens", "Graph amortized tokens / query", "Iterative total tokens"],
    [
        [n, BUILD_TOKENS + n * GRAPH_QUERY_TOKENS, BUILD_TOKENS / n + GRAPH_QUERY_TOKENS, n * ITERATIVE_QUERY_TOKENS]
        for n in QUERY_COUNTS
    ],
)

| Queries | Graph total tokens | Graph amortized tokens / query | Iterative total tokens |
| --- | --- | --- | --- |
| 10 | 1010000 | 101000.0000 | 40000 |
| 1000 | 2000000 | 2000.0000 | 4000000 |
| 10000 | 11000000 | 1100.0000 | 40000000 |

## 5. Query a prebuilt graph index on your model

The cell below calls `graph_examples`, one step behind `load_or_run`. With
`RUN_EXPENSIVE = False` it replays the reference: atlas-rag HippoRAG 2 answering
the first MuSiQue question of each hop category with `Qwen3_6-35B-A3B`, from
RI #2's release. With `RUN_EXPENSIVE = True` it queries the installed RI #4 index
on `MODELS` and caches the result. Choose `hipporag2` for **atlas-rag's HippoRAG 2
variant**, or `lightrag` for LightRAG in `mix` mode, which has no reference here
and always runs live. Both use the same cited final answerer and five original
passages.

Install the checksum-pinned indexes using the
[guide](GUIDE.md#compare-the-prebuilt-graph-indexes) before a live run. The runner
verifies the released bytes, full chunk coverage, and original-passage identities
before loading either backend. Missing vector stores never trigger index
construction. Query encoding uses the pinned local embedding model.

Live chat uses the shared model roles and Vector credential resolver. The
default family is Google; use `ADVANCED_RAG_MODEL_FAMILY=onprem` for Qwen
generation and extraction. Embeddings stay local. Set
`ADVANCED_RAG_SPEND_LIMIT_USD` before execution to limit spending. See the
[provider settings](GUIDE.md#models-credentials-and-spend). Recorded results
retain their original models; fresh comparisons require matched model roles.

In [8]:
# The reference covers HippoRAG 2 on the first question per hop category. LightRAG
# has no reference worked example, so it needs RUN_EXPENSIVE = True and the index.
from aieng.advanced_rag.runs import ReferenceMissingError
from synthesis_lab.reference import graph_examples


GRAPH_BACKEND = "hipporag2"
try:
    queried = graph_examples(backend=GRAPH_BACKEND, models=MODELS, live=RUN_EXPENSIVE)
except ReferenceMissingError:
    print(f"No {GRAPH_BACKEND} reference. Set RUN_EXPENSIVE = True and install RI #4's index to run it.")
else:
    show_table(
        ["Question", "Answer", "Stop", "Generation IDs", "Calls"],
        [
            [r["question"], r["answer"] or "(abstain)", r["stop_reason"], r["generation_ids"], r["usage"]["calls"]]
            for r in queried.value["records"]
        ],
    )

graph-examples reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
               6 calls, 8,156 prompt + 1,193 completion tokens
               rebuilding with gemini-3.5-flash-lite: about 6 calls, est. $0.01 (set RUN_EXPENSIVE = True to rebuild)


| Question | Answer | Stop | Generation IDs | Calls |
| --- | --- | --- | --- | --- |
| When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona? | June 1982 | answered | ['musique::00001', 'musique::00002', 'musique::00009', 'musique::00019', 'musique::00011'] | 2 |
| What month did the Tripartite discussions begin between Britain, France, and the country where, despite being headquartered in the nation called the nobilities commonwealth, the top-ranking Warsaw Pact operatives originated? | August | answered | ['musique::08421', 'musique::00031', 'musique::00025', 'musique::00942', 'musique::00938'] | 2 |
| When was the region immediately north of the region where Israel is located and the location of the Battle of Qurah and Umm al Maradim created? | Not stated in the provided sources. | the model reported the sources do not answer the question | ['musique::00110', 'musique::00108', 'musique::00105', 'musique::00104', 'musique::00545'] | 2 |

## 6. Interpretation

The toy isolates an evidence-selection mechanism. The actual graph comparison
still requires verified source mappings and matched execution before any quality
ranking is justified. A graph can recover a connecting passage and still give a
reader a distracting or unsupported context.

Before claiming a graph advantage, answer:

1. Did the gain arise from a better seed, a useful edge, or a larger evidence budget?
2. Are returned chunks mapped/deduplicated to the same original passage unit?
3. Which implementation and producer settings does the index actually represent?
4. At your query volume, how much build cost is amortized into each answer?

Next, [notebook 05](05_decision_grid.ipynb) compares these decisions across corpus
structure and query complexity and keeps unavailable cells explicit.

Keep gold answers and hop graphs with the scorer. Local pinned embeddings and
the Vector proxy are used for live RI #2 experiments. Source IDs establish
provenance; they do not by themselves establish factual entailment.

See the [participant guide](GUIDE.md), [corpus/provenance record](CORPUS.md),
[experiment contracts](docs/EXPERIMENTS.md), and [full matched comparison](docs/COMPARISONS.md).